In [26]:
import pygsheets # use 'pip install pygsheets', not maintained with conda

import pandas
import datetime
import numpy

# terminals

In [27]:
gc = pygsheets.authorize(service_account_env_var='GDRIVE_API_CREDENTIALS')
spreadsheet = gc.open_by_key('1tcS6Wd-Wp-LTDpLzFgJY_RSNDnbyubW3J_9HKIAys4A')
#spreadsheet = gc.open_by_key('1MrghwBeCz8Tzgua7CWGg_KoXKVZsV7r0kHMYHYqnNTg') # for July 2022 version update (terminals)

terms_df_orig = spreadsheet.worksheet('title', 'Terminals').get_as_df(start='A3')
terms_dict_df = spreadsheet.worksheet('title', 'Data dictionary').get_as_df()
terms_acronyms_df = spreadsheet.worksheet('title', 'Acronyms').get_as_df()
terms_copyright_df = spreadsheet.worksheet('title', 'Copyright').get_as_df()

/Users/baird/miniconda3/envs/gem/lib/python3.11/site-packages/pygsheets/worksheet.py:1554: UserWarning: At least one column name in the data frame is an empty string. If this is a concern, please specify include_tailing_empty=False and/or ensure that each column containing data has a name.
  warnings.warn('At least one column name in the data frame is an empty string. If this is a concern, please specify include_tailing_empty=False and/or ensure that each column containing data has a name.')


## clean up terminals

In [28]:
# remove oil export terminals
terms_df_orig = terms_df_orig[terms_df_orig['Fuel']=='LNG']
# remove anything without a wiki page
terms_df_orig = terms_df_orig[terms_df_orig['Wiki']!='']

## subset columns to include relevant columns

Note these columns are specified in the "Data dictionary" tab of the Pipelines_Current sheet; the column IncludeWithDataRelease has a Yes if the column is included, and DataReleaseColumnOrder determines how they're arranged.

In [29]:
terms_dict_df_include = terms_dict_df.copy()[terms_dict_df.copy()['IncludeWithDataRelease']=='Yes']
terms_dict_df_include = terms_dict_df_include.loc[
    terms_dict_df_include.DataReleaseColumnOrder!=''].sort_values(
    'DataReleaseColumnOrder', ascending=True)
terms_dict_df_include = terms_dict_df_include[['VariableName','Definition']]

In [30]:
terms_df_subset = terms_df_orig.copy()[terms_dict_df_include['VariableName'].tolist()]
terms_df_subset.sort_values('ComboID', inplace=True)

## write to Excel file

Use ExcelFormatter from Pandas to write 4 different tabs to the same file.

In [31]:
#pandas.io.formats.excel.ExcelFormatter.header_style = None

excel_writer = pandas.ExcelWriter('./DATA-FILES/GEM-LNG-Terminals-'+str(datetime.date.today())+'.xlsx')

terms_df_subset.to_excel(excel_writer, sheet_name='LNG Terminals '+str(datetime.date.today()), index=False)
terms_dict_df_include.to_excel(excel_writer, sheet_name='Data dictionary', index=False)
terms_acronyms_df.to_excel(excel_writer, sheet_name='Acronyms', index=False)
terms_copyright_df.to_excel(excel_writer, sheet_name='Copyright', index=False)

excel_writer.close()

# pipelines

In [49]:
#fuel_type = 'Gas'
fuel_type = 'Oil'
#fuel_type = 'Oil-and-Gas'

gc = pygsheets.authorize(service_account_env_var='GDRIVE_API_CREDENTIALS')
spreadsheet = gc.open_by_key('1foPLE6K-uqFlaYgLPAUxzeXfDO5wOOqE7tibNHeqTek')
#spreadsheet[1] "Gas Pipelines" tab is the second index
gas_pipes = spreadsheet.worksheet('title', 'Gas pipelines').get_as_df(start='A3')
oil_pipes = spreadsheet.worksheet('title', 'Oil/NGL pipelines').get_as_df(start='A3')
#owners = spreadsheet[3].get_as_df()

#remove liquid pipelines you don't want to distribute
remove_fuels = ['CO2']
oil_pipes = oil_pipes[~oil_pipes['Fuel'].isin(remove_fuels)]

pipes_acronyms_df = spreadsheet.worksheet('title', 'Acronyms').get_as_df()

if fuel_type == 'Gas':
    pipes_df_orig = gas_pipes.copy() #pandas.concat([oil_pipes, gas_pipes], ignore_index=True)
    pipes_dict_df = spreadsheet.worksheet('title', 'Data dictionary - Gas pipelines').get_as_df()
if fuel_type == 'Oil':
    pipes_df_orig = oil_pipes.copy()
    pipes_df_orig = pipes_df_orig.loc[pipes_df_orig.Fuel.isin(['Oil','NGL'])]
    pipes_dict_df = spreadsheet.worksheet('title', 'Data dictionary - Oil/NGL pipelines').get_as_df()
if fuel_type == 'Oil-and-Gas':
    pipes_df_orig = pandas.concat([oil_pipes, gas_pipes], ignore_index=True)

if fuel_type == 'Gas':
    pipes_copyright_df = spreadsheet.worksheet('title', 'Copyright - GGIT').get_as_df()
elif fuel_type == 'Oil':
    pipes_copyright_df = spreadsheet.worksheet('title', 'Copyright - GOIT').get_as_df()

/Users/baird/miniconda3/envs/gem/lib/python3.11/site-packages/pygsheets/worksheet.py:1554: UserWarning: At least one column name in the data frame is an empty string. If this is a concern, please specify include_tailing_empty=False and/or ensure that each column containing data has a name.
  warnings.warn('At least one column name in the data frame is an empty string. If this is a concern, please specify include_tailing_empty=False and/or ensure that each column containing data has a name.')


## clean up pipelines

In [50]:
# clean up rows that should not be distributed
pipes_df_orig = pipes_df_orig.loc[pipes_df_orig['Status']!='N/A']
pipes_df_orig = pipes_df_orig.loc[pipes_df_orig['PipelineName']!='']
#pipes_df_orig = pipes_df_orig.loc[pipes_df_orig.Wiki!='']

In [51]:
status_in_dev = ['proposed', 
                 'construction', 
                 'shelved', 'operating', 
                 'mothballed', 
                 'cancelled', 
                 'retired', 
                 'idle']
no_route_options = [
    'Unavailable', 
    'Capacity expansion only', 
    'Bidirectionality upgrade only',
    'Short route (< 100 km)', 
    'N/A',
    ''
]

# filter for the statuses above in the status_in_dev list (modify as desired)
#gas_pipes = gas_pipes[gas_pipes['Status'].str.lower().isin(status_in_dev)]

## subset pipeline columns to include relevant columns

Note these columns are specified in the "Data dictionary" tab of the Pipelines_Current sheet; the column IncludeWithDataRelease has a Yes if the column is included, and DataReleaseColumnOrder determines how they're arranged.

In [52]:
pipes_dict_df_include = pipes_dict_df.copy()[(pipes_dict_df.copy()['IncludeWithDataRelease']=='Yes')]
pipes_dict_df_include = pipes_dict_df_include.sort_values('DataReleaseColumnOrder', ascending=True)
pipes_dict_df_include = pipes_dict_df_include[['VariableName','Definition']]

In [53]:
pipes_df_subset = pipes_df_orig.copy()[pipes_dict_df_include['VariableName'].tolist()]

In [54]:
pipes_df_subset = pipes_df_subset.sort_values('PipelineName')

## write to Excel file

Use ExcelFormatter from Pandas to write 4 different tabs to the same file.

In [55]:
#pandas.io.formats.excel.ExcelFormatter.header_style = None

excel_writer = pandas.ExcelWriter('./DATA-FILES/GEM-'+fuel_type+'-Pipelines-'+str(datetime.date.today())+'.xlsx')

pipes_df_subset.to_excel(excel_writer, sheet_name='Pipelines', index=False)# '+str(datetime.date.today()), index=False)
pipes_dict_df_include.to_excel(excel_writer, sheet_name='Data dictionary', index=False)
pipes_acronyms_df.to_excel(excel_writer, sheet_name='Acronyms', index=False)
if fuel_type in ['Oil','Gas']:
    pipes_copyright_df.to_excel(excel_writer, sheet_name='Copyright', index=False)

#workbook = excel_writer.book
#worksheet = excel_writer.sheets['Terminals '+str(datetime.date.today())]
#format = workbook.add_format({'text_wrap': True})

excel_writer.close()